# 02 Graph Models
#
Proposal-aligned Phase 2 notebook.
#
This notebook does five things:
1. Install and verify PyTorch Geometric if needed
2. Load frozen Phase 1 artifacts
3. Build proposal-aligned heterogeneous transaction graphs
4. Train GCN, GraphSAGE, and Temporal GraphSAGE
5. Save raw graph outputs for later reporting scripts
#
Important design notes:
- We use explicit transaction nodes plus entity nodes
- We keep one unified PyG graph with node-type features
- This is heterogeneous in graph design, while staying practical on Kaggle
- This phase should save only raw scientific outputs, not thesis-facing figures/tables

In [1]:
import importlib.util
import subprocess
import sys

import torch

if importlib.util.find_spec("torch_geometric") is None:
    print("Installing PyTorch Geometric for this Kaggle runtime")
    torch_base = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"
    cuda_tag = "cpu" if torch.version.cuda is None else f"cu{torch.version.cuda.replace('.', '')}"
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_base}+{cuda_tag}.html"
    print("Using wheel index:", wheel_url)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch_geometric",
            "pyg_lib",
            "torch_scatter",
            "torch_sparse",
            "torch_cluster",
            "-f",
            wheel_url,
        ]
    )

Installing PyTorch Geometric for this Kaggle runtime
Using wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 96.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.7 MB/s eta 0:00:00


In [2]:
import gc
import json
import math
import os
import random
import time
import warnings
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GCNConv, SAGEConv

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## Cell 1: Settings
#
Update `PHASE1_INPUT_ROOT` to match your mounted Kaggle dataset path.

In [3]:
SEED = 42
PHASE1_INPUT_ROOT = Path("/kaggle/input/datasets/ashwinwalunj/ashwin-thesis-outputs/thesis_outputs")
OUTPUT_ROOT = Path("/kaggle/working/thesis_outputs")
PHASE2_ROOT = OUTPUT_ROOT / "artifacts" / "phase02_graph_models"

TARGET_COL = "isFraud"
ID_COL = "TransactionID"
TIME_COL = "TransactionDT"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

REFRESH_GRAPHS = False
REFRESH_MODELS = False

RUN_GRAPH_VARIANT_ABLATION = True
RUN_TEMPORAL_DECAY_ABLATION = True


GRAPH_MAX_GAP_DAYS = 30
DEFAULT_DECAY_LAMBDA = 0.02
MIN_ENTITY_FREQ = 2
MAX_ENTITY_FREQ = 20000
GNN_DROPOUT = 0.15
GNN_LR = 3e-4
GNN_WEIGHT_DECAY = 1e-5
GNN_EPOCHS = 20
GNN_PATIENCE = 5
FEATURE_CLIP_VALUE = 5.0
POS_WEIGHT_CAP = 10.0
LATENCY_BENCHMARK_REPEATS = 10
LATENCY_BENCHMARK_WARMUP = 2

NEIGHBOR_SIZES = [10, 5]
TRAIN_BATCH_SIZE = 512
EVAL_BATCH_SIZE = 1024
GNN_HIDDEN_DIM = 64


ANALYST_BUDGETS = [100, 250, 500, 1000, 2500, 5000]

for folder in [
    PHASE2_ROOT,
    OUTPUT_ROOT / "manifests",
]:
    folder.mkdir(parents=True, exist_ok=True)

## Cell 2: Reproducibility

In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

## Cell 3: Helper Functions

In [5]:
def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=str)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def recall_at_fpr(y_true, y_score, target_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    valid_idx = np.where(fpr <= target_fpr)[0]
    if len(valid_idx) == 0:
        return 0.0
    return float(tpr[valid_idx[-1]])


def recall_at_k(y_true, y_score, k):
    top_idx = np.argsort(-y_score)[:k]
    positives = max(int(np.sum(y_true)), 1)
    return float(np.sum(y_true[top_idx]) / positives)


def valid_entity_mask(series):
    return (
        series.notna()
        & (series.astype(str) != "nan")
        & (series.astype(str) != "__MISSING__")
        & (series.astype(str) != "__RARE__")
    )


def node_type_one_hot(node_type_id, num_types):
    eye = np.eye(num_types, dtype=np.float32)
    return eye[node_type_id]


def benchmark_score_latency_ms_per_transaction(score_fn, repeats=LATENCY_BENCHMARK_REPEATS, warmup=LATENCY_BENCHMARK_WARMUP):
    num_rows = None

    for _ in range(warmup):
        scores = score_fn()
        if num_rows is None:
            num_rows = max(len(scores), 1)

    timings = []
    for _ in range(repeats):
        start = perf_counter()
        scores = score_fn()
        end = perf_counter()
        if num_rows is None:
            num_rows = max(len(scores), 1)
        timings.append((end - start) * 1000.0 / num_rows)

    return {
        "mean_ms_per_transaction": float(np.mean(timings)),
        "median_ms_per_transaction": float(np.median(timings)),
        "std_ms_per_transaction": float(np.std(timings)),
        "repeats": int(repeats),
        "rows_benchmarked": int(num_rows),
    }


def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Cell 4: Load Frozen Phase 1 Artifacts

In [6]:
PHASE1_ARTIFACTS = PHASE1_INPUT_ROOT / "artifacts" / "phase01_data_baselines"

required_files = [
    PHASE1_ARTIFACTS / "merged_train.parquet",
    PHASE1_ARTIFACTS / "train.parquet",
    PHASE1_ARTIFACTS / "valid.parquet",
    PHASE1_ARTIFACTS / "test.parquet",
    PHASE1_ARTIFACTS / "feature_schema.json",
    PHASE1_ARTIFACTS / "baseline_predictions.parquet",
    PHASE1_ARTIFACTS / "temporal_oof_folds.json",
    PHASE1_ARTIFACTS / "baseline_metrics.json",
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing Phase 1 file: {file_path}")

raw_merged_df = pd.read_parquet(PHASE1_ARTIFACTS / "merged_train.parquet")
train_df = pd.read_parquet(PHASE1_ARTIFACTS / "train.parquet")
valid_df = pd.read_parquet(PHASE1_ARTIFACTS / "valid.parquet")
test_df = pd.read_parquet(PHASE1_ARTIFACTS / "test.parquet")
feature_schema = load_json(PHASE1_ARTIFACTS / "feature_schema.json")
phase1_predictions = pd.read_parquet(PHASE1_ARTIFACTS / "baseline_predictions.parquet")
oof_folds = load_json(PHASE1_ARTIFACTS / "temporal_oof_folds.json")
phase1_metrics = load_json(PHASE1_ARTIFACTS / "baseline_metrics.json")

print("Phase 1 loaded")
print("Raw merged:", raw_merged_df.shape)
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

Phase 1 loaded
Raw merged: (590540, 434)
Train: (442905, 479)
Valid: (59054, 479)
Test : (88581, 479)


## Cell 5: Prepare Proposal-Aligned Graph Source Frame
#
This frame keeps the raw entity values required for heterogeneous graph construction.

In [7]:
graph_df = raw_merged_df.copy().sort_values(TIME_COL).reset_index(drop=True)
del raw_merged_df
release_memory()
graph_df["txn_idx"] = np.arange(len(graph_df), dtype=np.int64)

graph_df["split"] = "train"
graph_df.loc[len(train_df): len(train_df) + len(valid_df) - 1, "split"] = "valid"
graph_df.loc[len(train_df) + len(valid_df):, "split"] = "test"

graph_df["dt_hour"] = ((graph_df[TIME_COL] // 3600) % 24).astype("int16")
graph_df["dt_dayofweek"] = ((graph_df[TIME_COL] // 86400) % 7).astype("int16")
graph_df["dt_week"] = (graph_df[TIME_COL] // (86400 * 7)).astype("int32")

graph_df["card_identity_key"] = np.where(
    graph_df["card1"].notna(),
    graph_df[["card1", "card2", "card3", "card5"]].fillna("na").astype(str).agg("_".join, axis=1),
    np.nan,
)

graph_df["device_key"] = np.where(
    graph_df["DeviceType"].notna() | graph_df["DeviceInfo"].notna(),
    graph_df[["DeviceType", "DeviceInfo"]].fillna("na").astype(str).agg("_".join, axis=1),
    np.nan,
)

graph_df["payer_email_key"] = np.where(
    graph_df["P_emaildomain"].notna(),
    "P_" + graph_df["P_emaildomain"].astype(str),
    np.nan,
)
graph_df["receiver_email_key"] = np.where(
    graph_df["R_emaildomain"].notna(),
    "R_" + graph_df["R_emaildomain"].astype(str),
    np.nan,
)

# IEEE-CIS does not expose a real merchant id.
# We use a documented merchant proxy from transaction context.
graph_df["merchant_proxy_key"] = np.where(
    graph_df["ProductCD"].notna() & graph_df["addr1"].notna() & graph_df["addr2"].notna(),
    graph_df[["ProductCD", "addr1", "addr2"]].astype(str).agg("_".join, axis=1),
    np.nan,
)

train_amount = graph_df.loc[graph_df["split"] == "train", "TransactionAmt"].fillna(0).clip(lower=0)
train_amount_log = np.log1p(train_amount)
amount_low = float(train_amount_log.quantile(0.01))
amount_high = float(train_amount_log.quantile(0.99))
if amount_high <= amount_low:
    amount_high = amount_low + 1.0

graph_df["transaction_amt_log1p"] = np.log1p(graph_df["TransactionAmt"].fillna(0).clip(lower=0))
graph_df["amount_norm"] = (
    (graph_df["transaction_amt_log1p"] - amount_low) / (amount_high - amount_low)
).clip(0.0, 1.0)
graph_df["amount_norm"] = (0.1 + 0.9 * graph_df["amount_norm"]).astype("float32")

graph_df[[ID_COL, TIME_COL, "split", "card_identity_key", "device_key", "merchant_proxy_key", "amount_norm"]].head()

,TransactionID,TransactionDT,split,card_identity_key,device_key,merchant_proxy_key,amount_norm
0,2987000,86400,train,13926_na_150.0_142.0,NaN,W_315.0_87.0,0.463845
1,2987001,86401,train,2755_404.0_150.0_102.0,NaN,W_325.0_87.0,0.300746
2,2987002,86469,train,4663_490.0_150.0_166.0,NaN,W_330.0_87.0,0.435311
3,2987003,86499,train,18132_567.0_150.0_117.0,NaN,W_476.0_87.0,0.403760
4,2987004,86506,train,4497_514.0_150.0_102.0,mobile_SAMSUNG SM-G892A Build/NRD90M,H_420.0_87.0,0.403760


## Cell 6: Node Feature Base
#
Transaction nodes reuse frozen Phase 1 features. Entity nodes receive train-only entity statistics.

In [8]:
full_processed_df = pd.concat([train_df, valid_df, test_df], axis=0, ignore_index=True)

if len(full_processed_df) != len(graph_df):
    raise ValueError("Phase 1 processed frame and graph source frame are misaligned.")

feature_columns = feature_schema["feature_columns"]
transaction_feature_matrix_raw = full_processed_df[feature_columns].astype(np.float32).to_numpy()
transaction_labels = full_processed_df[TARGET_COL].astype(np.int64).to_numpy()

# Neural graph models are much more sensitive to feature scale than the tree models in Phase 1.
# We fit scaling on the training transactions only, then apply it to every split.
train_feature_block = transaction_feature_matrix_raw[: len(train_df)]
feature_mean = train_feature_block.mean(axis=0)
feature_std = train_feature_block.std(axis=0)
feature_std[feature_std < 1e-6] = 1.0
transaction_feature_matrix = (transaction_feature_matrix_raw - feature_mean) / feature_std
transaction_feature_matrix = np.clip(transaction_feature_matrix, -FEATURE_CLIP_VALUE, FEATURE_CLIP_VALUE).astype(np.float32)

transaction_time_norm = (graph_df[TIME_COL].to_numpy().astype(np.float32) / 86400.0).reshape(-1, 1)
transaction_time_norm = (transaction_time_norm - transaction_time_norm.mean()) / (transaction_time_norm.std() + 1e-6)
transaction_hour_sin = np.sin(2 * np.pi * graph_df["dt_hour"].to_numpy(dtype=np.float32) / 24.0).reshape(-1, 1)
transaction_hour_cos = np.cos(2 * np.pi * graph_df["dt_hour"].to_numpy(dtype=np.float32) / 24.0).reshape(-1, 1)
transaction_time_extra = np.concatenate([transaction_time_norm, transaction_hour_sin, transaction_hour_cos], axis=1).astype(np.float32)
del full_processed_df
release_memory()

print("Transaction feature shape:", transaction_feature_matrix.shape)

Transaction feature shape: (590540, 477)


## Cell 7: Graph Variant Definitions

In [9]:
graph_variant_specs = {
    "cards": [
        {"type_name": "card", "column": "card_identity_key"},
    ],
    "cards_plus_merchants": [
        {"type_name": "card", "column": "card_identity_key"},
        {"type_name": "merchant_proxy", "column": "merchant_proxy_key"},
    ],
    "full": [
        {"type_name": "card", "column": "card_identity_key"},
        {"type_name": "device", "column": "device_key"},
        {"type_name": "email", "column": "payer_email_key"},
        {"type_name": "email", "column": "receiver_email_key"},
        {"type_name": "merchant_proxy", "column": "merchant_proxy_key"},
    ],
}

GRAPH_DISPLAY_NAMES = {
    "cards": "Cards",
    "cards_plus_merchants": "Cards+Merchants",
    "full": "Full",
}

NODE_TYPE_TO_ID = {
    "transaction": 0,
    "card": 1,
    "device": 2,
    "email": 3,
    "merchant_proxy": 4,
}

## Cell 8: Heterogeneous Graph Builder
#
Proposal-aligned edge weighting:
w = exp(-lambda * delta_t) * normalized_amount

In [10]:
def build_heterogeneous_graph(graph_source_df, variant_name, relation_specs, decay_lambda):
    entity_node_maps = {}
    entity_stats_frames = {}
    edge_rows = []
    next_node_id = len(graph_source_df)

    train_only = graph_source_df[graph_source_df["split"] == "train"].copy()

    for spec in relation_specs:
        entity_type = spec["type_name"]
        column = spec["column"]

        work = graph_source_df[[ID_COL, "txn_idx", TIME_COL, "amount_norm", TARGET_COL, column]].copy()
        work = work[valid_entity_mask(work[column])].copy()
        if work.empty:
            continue

        work[column] = work[column].astype(str)
        entity_counts = work[column].value_counts()
        keep_entities = entity_counts[(entity_counts >= MIN_ENTITY_FREQ) & (entity_counts <= MAX_ENTITY_FREQ)].index
        work = work[work[column].isin(keep_entities)].copy()
        if work.empty:
            continue
        prefixed_entity = entity_type + "::" + work[column]

        unique_entities = pd.Index(prefixed_entity.unique())
        entity_map = {entity_value: node_id for node_id, entity_value in enumerate(unique_entities, start=next_node_id)}
        next_node_id += len(unique_entities)
        entity_node_maps[(entity_type, column)] = entity_map

        train_stats = train_only[[column, "TransactionAmt", TARGET_COL]].copy()
        train_stats = train_stats[valid_entity_mask(train_stats[column])].copy()
        train_stats[column] = train_stats[column].astype(str)
        train_stats = train_stats[train_stats[column].isin(keep_entities)].copy()
        train_stats[column] = entity_type + "::" + train_stats[column]
        train_stats["amt_log1p"] = np.log1p(train_stats["TransactionAmt"].fillna(0).clip(lower=0))
        stats_df = (
            train_stats.groupby(column)
            .agg(
                train_count=(column, "size"),
                train_amount_mean=("amt_log1p", "mean"),
                train_fraud_rate=(TARGET_COL, "mean"),
            )
            .reset_index()
            .rename(columns={column: "entity_key"})
        )
        entity_stats_frames[(entity_type, column)] = stats_df

        work["entity_key"] = prefixed_entity
        work["entity_node_id"] = work["entity_key"].map(entity_map)
        work = work.sort_values(["entity_key", TIME_COL, "txn_idx"]).reset_index(drop=True)
        work["prev_time"] = work.groupby("entity_key")[TIME_COL].shift(1)
        work["delta_days"] = ((work[TIME_COL] - work["prev_time"]) / 86400.0).fillna(0.0)
        work["delta_days"] = work["delta_days"].clip(lower=0.0, upper=GRAPH_MAX_GAP_DAYS)
        work["temporal_weight"] = np.exp(-decay_lambda * work["delta_days"].to_numpy(dtype=np.float32))
        work["edge_weight"] = (work["temporal_weight"] * work["amount_norm"]).astype(np.float32)
        work["edge_type"] = entity_type

        forward_edges = pd.DataFrame(
            {
                "src": work["txn_idx"].to_numpy(dtype=np.int64),
                "dst": work["entity_node_id"].to_numpy(dtype=np.int64),
                "weight": work["edge_weight"].to_numpy(dtype=np.float32),
                "edge_type": work["edge_type"],
            }
        )
        backward_edges = pd.DataFrame(
            {
                "src": work["entity_node_id"].to_numpy(dtype=np.int64),
                "dst": work["txn_idx"].to_numpy(dtype=np.int64),
                "weight": work["edge_weight"].to_numpy(dtype=np.float32),
                "edge_type": work["edge_type"],
            }
        )
        edge_rows.append(forward_edges)
        edge_rows.append(backward_edges)

    if not edge_rows:
        raise ValueError(f"No edges were built for variant {variant_name}.")

    edge_df = pd.concat(edge_rows, axis=0, ignore_index=True)

    entity_rows = []
    for (entity_type, column), entity_map in entity_node_maps.items():
        stats_df = entity_stats_frames[(entity_type, column)]
        stats_lookup = stats_df.set_index("entity_key")
        for entity_key, node_id in entity_map.items():
            if entity_key in stats_lookup.index:
                row = stats_lookup.loc[entity_key]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                train_count = float(row["train_count"])
                train_amount_mean = float(row["train_amount_mean"])
                train_fraud_rate = float(row["train_fraud_rate"])
            else:
                train_count = 0.0
                train_amount_mean = 0.0
                train_fraud_rate = 0.0
            entity_rows.append(
                {
                    "node_id": node_id,
                    "entity_key": entity_key,
                    "entity_type": entity_type,
                    "train_count": train_count,
                    "train_amount_mean": train_amount_mean,
                    "train_fraud_rate": train_fraud_rate,
                }
            )

    entity_df = pd.DataFrame(entity_rows).sort_values("node_id").reset_index(drop=True)
    entity_df["train_count_norm"] = np.log1p(entity_df["train_count"].fillna(0))
    if entity_df["train_count_norm"].max() > 0:
        entity_df["train_count_norm"] = entity_df["train_count_norm"] / entity_df["train_count_norm"].max()

    return edge_df, entity_df

## Cell 9: Build Variant Graphs

In [11]:
variant_graphs = {}
variant_metadata_rows = []

for variant_name, relation_specs in graph_variant_specs.items():
    edge_file = PHASE2_ROOT / f"{variant_name}_edges.parquet"
    entity_file = PHASE2_ROOT / f"{variant_name}_entity_nodes.parquet"
    meta_file = PHASE2_ROOT / f"{variant_name}_metadata.json"

    if edge_file.exists() and entity_file.exists() and meta_file.exists() and not REFRESH_GRAPHS:
        print(f"Loading cached graph variant: {variant_name}")
        metadata = load_json(meta_file)
    else:
        print(f"Building graph variant: {variant_name}")
        edge_df, entity_df = build_heterogeneous_graph(
            graph_source_df=graph_df,
            variant_name=variant_name,
            relation_specs=relation_specs,
            decay_lambda=DEFAULT_DECAY_LAMBDA,
        )
        edge_df.to_parquet(edge_file, index=False)
        entity_df.to_parquet(entity_file, index=False)

        degree_series = edge_df.groupby("src").size()
        metadata = {
            "variant_name": variant_name,
            "num_transaction_nodes": int(len(graph_df)),
            "num_entity_nodes": int(len(entity_df)),
            "num_nodes_total": int(len(graph_df) + len(entity_df)),
            "num_edges": int(len(edge_df)),
            "avg_degree": float(degree_series.mean()),
            "median_degree": float(degree_series.median()),
            "max_degree": int(degree_series.max()),
            "entity_types": sorted(entity_df["entity_type"].unique().tolist()),
            "decay_lambda": DEFAULT_DECAY_LAMBDA,
        }
        save_json(metadata, meta_file)

    variant_graphs[variant_name] = {
        "edge_path": str(edge_file),
        "entity_path": str(entity_file),
        "metadata_path": str(meta_file),
    }
    variant_metadata_rows.append(metadata)
    if "edge_df" in locals():
        del edge_df
    if "entity_df" in locals():
        del entity_df
    release_memory()

graph_metadata_df = pd.DataFrame(variant_metadata_rows).sort_values("variant_name")
graph_metadata_df.to_csv(PHASE2_ROOT / "graph_variant_metadata.csv", index=False)
graph_metadata_df

Building graph variant: cards
Building graph variant: cards_plus_merchants
Building graph variant: full


,variant_name,num_transaction_nodes,num_entity_nodes,num_nodes_total,num_edges,avg_degree,median_degree,max_degree,entity_types,decay_lambda
0,cards,590540,10765,601305,1172920,1.963950,1.0,14112,[card],0.02
1,cards_plus_merchants,590540,11198,601738,1892198,3.151457,2.0,18409,"[card, merchant_proxy]",0.02
2,full,590540,12745,603285,2254186,3.740847,2.0,19781,"[card, device, email, merchant_proxy]",0.02


## Cell 10: Build Unified PyG Data Objects

In [12]:
def load_variant_graph_frames(variant_name):
    edge_df = pd.read_parquet(variant_graphs[variant_name]["edge_path"])
    entity_df = pd.read_parquet(variant_graphs[variant_name]["entity_path"])
    return edge_df, entity_df


def build_data_object_for_variant(variant_name, feature_mode):
    edge_df, entity_df = load_variant_graph_frames(variant_name)

    num_transaction_nodes = len(graph_df)
    num_entity_nodes = len(entity_df)
    num_total_nodes = num_transaction_nodes + num_entity_nodes

    base_dim = transaction_feature_matrix.shape[1]
    entity_stat_dim = 3
    node_type_dim = len(NODE_TYPE_TO_ID)
    time_dim = 3

    if feature_mode == "base":
        x = np.zeros((num_total_nodes, base_dim + entity_stat_dim + node_type_dim), dtype=np.float32)
    else:
        x = np.zeros((num_total_nodes, base_dim + time_dim + entity_stat_dim + node_type_dim), dtype=np.float32)

    transaction_type_oh = node_type_one_hot(NODE_TYPE_TO_ID["transaction"], node_type_dim)
    for idx in range(num_transaction_nodes):
        if feature_mode == "base":
            x[idx, :base_dim] = transaction_feature_matrix[idx]
            x[idx, base_dim + entity_stat_dim:] = transaction_type_oh
        else:
            x[idx, :base_dim] = transaction_feature_matrix[idx]
            x[idx, base_dim: base_dim + time_dim] = transaction_time_extra[idx]
            x[idx, base_dim + time_dim + entity_stat_dim:] = transaction_type_oh

    entity_stat_block = entity_df[["train_count_norm", "train_amount_mean", "train_fraud_rate"]].fillna(0).astype(np.float32)
    entity_stat_mean = entity_stat_block.mean(axis=0).to_numpy(dtype=np.float32)
    entity_stat_std = entity_stat_block.std(axis=0).to_numpy(dtype=np.float32)
    entity_stat_std[entity_stat_std < 1e-6] = 1.0

    for _, row in entity_df.iterrows():
        node_id = int(row["node_id"])
        node_type_oh = node_type_one_hot(NODE_TYPE_TO_ID[row["entity_type"]], node_type_dim)
        entity_stats = np.array(
            [
                row["train_count_norm"],
                row["train_amount_mean"],
                row["train_fraud_rate"],
            ],
            dtype=np.float32,
        )
        entity_stats = (entity_stats - entity_stat_mean) / entity_stat_std
        entity_stats = np.clip(entity_stats, -FEATURE_CLIP_VALUE, FEATURE_CLIP_VALUE)
        if feature_mode == "base":
            x[node_id, base_dim: base_dim + entity_stat_dim] = entity_stats
            x[node_id, base_dim + entity_stat_dim:] = node_type_oh
        else:
            x[node_id, base_dim + time_dim: base_dim + time_dim + entity_stat_dim] = entity_stats
            x[node_id, base_dim + time_dim + entity_stat_dim:] = node_type_oh

    y_all = np.full(num_total_nodes, -1, dtype=np.int64)
    y_all[:num_transaction_nodes] = transaction_labels

    train_mask = torch.zeros(num_total_nodes, dtype=torch.bool)
    valid_mask = torch.zeros(num_total_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_total_nodes, dtype=torch.bool)
    train_mask[: len(train_df)] = True
    valid_mask[len(train_df): len(train_df) + len(valid_df)] = True
    test_mask[len(train_df) + len(valid_df): num_transaction_nodes] = True

    edge_index = np.ascontiguousarray(edge_df[["src", "dst"]].to_numpy().T, dtype=np.int64)
    edge_weight = np.ascontiguousarray(edge_df["weight"].to_numpy(dtype=np.float32))
    data_obj = Data(
        x=torch.from_numpy(x),
        y=torch.from_numpy(y_all),
        edge_index=torch.from_numpy(edge_index),
        edge_weight=torch.from_numpy(edge_weight),
        train_mask=train_mask,
        valid_mask=valid_mask,
        test_mask=test_mask,
    )
    del edge_df, entity_df, entity_stat_block, x, y_all, edge_index, edge_weight
    release_memory()
    return data_obj


def build_data_object_from_subset(subset_graph_df, subset_processed_df, variant_name, feature_mode, train_rows_count):
    subset_graph_df = subset_graph_df.copy().reset_index(drop=True)
    subset_graph_df["txn_idx"] = np.arange(len(subset_graph_df), dtype=np.int64)

    relation_specs = graph_variant_specs[variant_name]
    edge_df, entity_df = build_heterogeneous_graph(
        graph_source_df=subset_graph_df,
        variant_name=variant_name,
        relation_specs=relation_specs,
        decay_lambda=DEFAULT_DECAY_LAMBDA,
    )

    subset_feature_matrix_raw = subset_processed_df[feature_columns].astype(np.float32).to_numpy()
    subset_labels = subset_processed_df[TARGET_COL].astype(np.int64).to_numpy()

    train_feature_block = subset_feature_matrix_raw[:train_rows_count]
    feature_mean = train_feature_block.mean(axis=0)
    feature_std = train_feature_block.std(axis=0)
    feature_std[feature_std < 1e-6] = 1.0
    subset_feature_matrix = (subset_feature_matrix_raw - feature_mean) / feature_std
    subset_feature_matrix = np.clip(subset_feature_matrix, -FEATURE_CLIP_VALUE, FEATURE_CLIP_VALUE).astype(np.float32)

    subset_time_norm = (subset_graph_df[TIME_COL].to_numpy().astype(np.float32) / 86400.0).reshape(-1, 1)
    subset_time_norm = (subset_time_norm - subset_time_norm.mean()) / (subset_time_norm.std() + 1e-6)
    subset_hour_sin = np.sin(2 * np.pi * subset_graph_df["dt_hour"].to_numpy(dtype=np.float32) / 24.0).reshape(-1, 1)
    subset_hour_cos = np.cos(2 * np.pi * subset_graph_df["dt_hour"].to_numpy(dtype=np.float32) / 24.0).reshape(-1, 1)
    subset_time_extra = np.concatenate([subset_time_norm, subset_hour_sin, subset_hour_cos], axis=1).astype(np.float32)

    num_transaction_nodes = len(subset_graph_df)
    num_entity_nodes = len(entity_df)
    num_total_nodes = num_transaction_nodes + num_entity_nodes

    base_dim = subset_feature_matrix.shape[1]
    entity_stat_dim = 3
    node_type_dim = len(NODE_TYPE_TO_ID)
    time_dim = 3

    if feature_mode == "base":
        x = np.zeros((num_total_nodes, base_dim + entity_stat_dim + node_type_dim), dtype=np.float32)
    else:
        x = np.zeros((num_total_nodes, base_dim + time_dim + entity_stat_dim + node_type_dim), dtype=np.float32)

    transaction_type_oh = node_type_one_hot(NODE_TYPE_TO_ID["transaction"], node_type_dim)
    for idx in range(num_transaction_nodes):
        if feature_mode == "base":
            x[idx, :base_dim] = subset_feature_matrix[idx]
            x[idx, base_dim + entity_stat_dim:] = transaction_type_oh
        else:
            x[idx, :base_dim] = subset_feature_matrix[idx]
            x[idx, base_dim : base_dim + time_dim] = subset_time_extra[idx]
            x[idx, base_dim + time_dim + entity_stat_dim :] = transaction_type_oh

    entity_stat_block = entity_df[["train_count_norm", "train_amount_mean", "train_fraud_rate"]].fillna(0).astype(np.float32)
    entity_stat_mean = entity_stat_block.mean(axis=0).to_numpy(dtype=np.float32)
    entity_stat_std = entity_stat_block.std(axis=0).to_numpy(dtype=np.float32)
    entity_stat_std[entity_stat_std < 1e-6] = 1.0

    for _, row in entity_df.iterrows():
        node_id = int(row["node_id"])
        node_type_oh = node_type_one_hot(NODE_TYPE_TO_ID[row["entity_type"]], node_type_dim)
        entity_stats = np.array(
            [row["train_count_norm"], row["train_amount_mean"], row["train_fraud_rate"]],
            dtype=np.float32,
        )
        entity_stats = (entity_stats - entity_stat_mean) / entity_stat_std
        entity_stats = np.clip(entity_stats, -FEATURE_CLIP_VALUE, FEATURE_CLIP_VALUE)
        if feature_mode == "base":
            x[node_id, base_dim : base_dim + entity_stat_dim] = entity_stats
            x[node_id, base_dim + entity_stat_dim :] = node_type_oh
        else:
            x[node_id, base_dim + time_dim : base_dim + time_dim + entity_stat_dim] = entity_stats
            x[node_id, base_dim + time_dim + entity_stat_dim :] = node_type_oh
    y_all = np.full(num_total_nodes, -1, dtype=np.int64)
    y_all[:num_transaction_nodes] = subset_labels

    train_mask = torch.zeros(num_total_nodes, dtype=torch.bool)
    valid_mask = torch.zeros(num_total_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_total_nodes, dtype=torch.bool)
    train_mask[:train_rows_count] = True
    valid_mask[train_rows_count:num_transaction_nodes] = True

    edge_index = np.ascontiguousarray(edge_df[["src", "dst"]].to_numpy().T, dtype=np.int64)
    edge_weight = np.ascontiguousarray(edge_df["weight"].to_numpy(dtype=np.float32))
    data_obj = Data(
        x=torch.from_numpy(x),
        y=torch.from_numpy(y_all),
        edge_index=torch.from_numpy(edge_index),
        edge_weight=torch.from_numpy(edge_weight),
        train_mask=train_mask,
        valid_mask=valid_mask,
        test_mask=test_mask,
    )
    del entity_stat_block, x, y_all, edge_index, edge_weight
    release_memory()
    return data_obj, edge_df, entity_df


data_objects = {}
full_data_base = build_data_object_for_variant("full", "base")
full_data_temporal = build_data_object_for_variant("full", "temporal")
data_objects["full"] = {"base": full_data_base, "temporal": full_data_temporal}
print("full", full_data_base)
release_memory()

full Data(x=[603285, 485], edge_index=[2, 2254186], y=[603285], edge_weight=[2254186], train_mask=[603285], valid_mask=[603285], test_mask=[603285])


## Cell 11: Neighbor Loaders

In [13]:
def make_loaders(data_obj):
    train_loader = NeighborLoader(
        data_obj,
        input_nodes=data_obj.train_mask,
        num_neighbors=NEIGHBOR_SIZES,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
    )
    valid_loader = NeighborLoader(
        data_obj,
        input_nodes=data_obj.valid_mask,
        num_neighbors=NEIGHBOR_SIZES,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
    )
    test_loader = NeighborLoader(
        data_obj,
        input_nodes=data_obj.test_mask,
        num_neighbors=NEIGHBOR_SIZES,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
    )
    return train_loader, valid_loader, test_loader

## Cell 12: Graph Models

In [14]:
class GCNModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.dropout = dropout
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_weight=None):
        x = self.input_proj(x)
        x = self.input_norm(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.out(x).view(-1)


class GraphSAGEModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)
        self.skip_proj = nn.Linear(in_dim, hidden_dim)
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = dropout
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_weight=None):
        skip = self.skip_proj(x)
        x = self.input_proj(x)
        x = self.input_norm(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = x + skip
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.out(x).view(-1)


class TemporalGraphSAGEModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        self.base_in_dim = in_dim - 3
        self.time_encoder = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
        )
        self.input_proj = nn.Linear(self.base_in_dim + 16, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)
        self.skip_proj = nn.Linear(self.base_in_dim, hidden_dim)
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = dropout
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_weight=None):
        base_feats = x[:, : self.base_in_dim]
        time_feats = x[:, self.base_in_dim : self.base_in_dim + 3]
        time_emb = self.time_encoder(time_feats)
        skip = self.skip_proj(base_feats)
        x = torch.cat([base_feats, time_emb], dim=1)
        x = self.input_proj(x)
        x = self.input_norm(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = x + skip
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.out(x).view(-1)

## Cell 13: Training And Evaluation Functions

In [15]:
train_labels_only = transaction_labels[: len(train_df)]
raw_pos_weight_value = (len(train_labels_only) - train_labels_only.sum()) / max(train_labels_only.sum(), 1)
pos_weight_value = min(raw_pos_weight_value, POS_WEIGHT_CAP)
pos_weight_tensor = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
print("Raw pos_weight:", round(float(raw_pos_weight_value), 3))
print("Clipped pos_weight for GNN training:", round(float(pos_weight_value), 3))


def make_model(model_name, in_dim):
    if model_name == "GCN":
        return GCNModel(in_dim, GNN_HIDDEN_DIM, GNN_DROPOUT).to(DEVICE)
    if model_name == "GraphSAGE":
        return GraphSAGEModel(in_dim, GNN_HIDDEN_DIM, GNN_DROPOUT).to(DEVICE)
    if model_name == "TemporalGraphSAGE":
        return TemporalGraphSAGEModel(in_dim, GNN_HIDDEN_DIM, GNN_DROPOUT).to(DEVICE)
    raise ValueError(f"Unknown model: {model_name}")


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch in loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, getattr(batch, "edge_weight", None))
        logits = logits[: batch.batch_size]
        y_batch = batch.y[: batch.batch_size].float()
        loss = F.binary_cross_entropy_with_logits(logits, y_batch, pos_weight=pos_weight_tensor)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += float(loss.item()) * batch.batch_size
        total_examples += int(batch.batch_size)

    return total_loss / max(total_examples, 1)


@torch.no_grad()
def collect_model_scores(model, loader):
    model.eval()
    all_probs = []
    all_true = []

    for batch in loader:
        batch = batch.to(DEVICE)
        logits = model(batch.x, batch.edge_index, getattr(batch, "edge_weight", None))
        logits = logits[: batch.batch_size]
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        y_batch = batch.y[: batch.batch_size].detach().cpu().numpy()
        all_probs.append(probs)
        all_true.append(y_batch)

    y_true = np.concatenate(all_true)
    y_score = np.concatenate(all_probs)
    return y_true, y_score


@torch.no_grad()
def evaluate_model(model, loader):
    y_true, y_score = collect_model_scores(model, loader)
    return {
        "auc_pr": float(average_precision_score(y_true, y_score)),
        "auc_roc": float(roc_auc_score(y_true, y_score)),
        "recall_at_1pct_fpr": float(recall_at_fpr(y_true, y_score, 0.01)),
        "y_true": y_true,
        "y_score": y_score,
    }


def fit_model(model_name, variant_name, data_obj):
    train_loader, valid_loader, test_loader = make_loaders(data_obj)
    model = make_model(model_name, data_obj.x.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=GNN_LR, weight_decay=GNN_WEIGHT_DECAY)

    best_valid_ap = -1.0
    best_epoch = 0
    stale_epochs = 0
    best_state = None
    history_rows = []

    for epoch in range(1, GNN_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        valid_metrics = evaluate_model(model, valid_loader)
        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_auc_pr": valid_metrics["auc_pr"],
                "valid_auc_roc": valid_metrics["auc_roc"],
                "valid_recall_at_1pct_fpr": valid_metrics["recall_at_1pct_fpr"],
            }
        )
        print(
            f"{variant_name} | {model_name} | epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | valid_auc_pr={valid_metrics['auc_pr']:.4f}"
        )

        if valid_metrics["auc_pr"] > best_valid_ap:
            best_valid_ap = valid_metrics["auc_pr"]
            best_epoch = epoch
            stale_epochs = 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            stale_epochs += 1

        if stale_epochs >= GNN_PATIENCE:
            print(f"Early stopping {model_name} on {variant_name} at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    valid_metrics = evaluate_model(model, valid_loader)
    test_metrics = evaluate_model(model, test_loader)
    history_df = pd.DataFrame(history_rows)

    latency_result = benchmark_score_latency_ms_per_transaction(
        lambda: collect_model_scores(model, test_loader)[1]
    )

    checkpoint_path = PHASE2_ROOT / f"{variant_name}_{model_name}_best.pt"
    torch.save(best_state, checkpoint_path)

    valid_scores = valid_metrics["y_score"]
    test_scores = test_metrics["y_score"]
    valid_txn = graph_df.loc[graph_df["split"] == "valid", [ID_COL, TIME_COL]].copy().reset_index(drop=True)
    test_txn = graph_df.loc[graph_df["split"] == "test", [ID_COL, TIME_COL]].copy().reset_index(drop=True)

    preds_df = pd.concat(
        [
            pd.DataFrame(
                {
                    "split": "valid",
                    ID_COL: valid_txn[ID_COL].to_numpy(),
                    TIME_COL: valid_txn[TIME_COL].to_numpy(),
                    f"{variant_name}_{model_name}_score": valid_scores,
                }
            ),
            pd.DataFrame(
                {
                    "split": "test",
                    ID_COL: test_txn[ID_COL].to_numpy(),
                    TIME_COL: test_txn[TIME_COL].to_numpy(),
                    f"{variant_name}_{model_name}_score": test_scores,
                }
            ),
        ],
        ignore_index=True,
    )

    summary = {
        "variant_name": variant_name,
        "model_name": model_name,
        "best_epoch": int(best_epoch),
        "valid_auc_pr": float(valid_metrics["auc_pr"]),
        "valid_auc_roc": float(valid_metrics["auc_roc"]),
        "valid_recall_at_1pct_fpr": float(valid_metrics["recall_at_1pct_fpr"]),
        "test_auc_pr": float(test_metrics["auc_pr"]),
        "test_auc_roc": float(test_metrics["auc_roc"]),
        "test_recall_at_1pct_fpr": float(test_metrics["recall_at_1pct_fpr"]),
        "test_latency_ms": float(latency_result["mean_ms_per_transaction"]),
        "checkpoint_path": str(checkpoint_path),
    }
    del train_loader, valid_loader, test_loader, model, optimizer, best_state
    release_memory()
    return history_df, preds_df, summary


def fit_oof_graph_model(model_name, variant_name, data_obj):
    train_loader, valid_loader, _ = make_loaders(data_obj)
    model = make_model(model_name, data_obj.x.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=GNN_LR, weight_decay=GNN_WEIGHT_DECAY)

    best_valid_ap = -1.0
    best_epoch = 0
    stale_epochs = 0
    best_state = None
    history_rows = []

    for epoch in range(1, GNN_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        valid_metrics = evaluate_model(model, valid_loader)
        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_auc_pr": valid_metrics["auc_pr"],
                "valid_auc_roc": valid_metrics["auc_roc"],
                "valid_recall_at_1pct_fpr": valid_metrics["recall_at_1pct_fpr"],
            }
        )
        print(
            f"{variant_name} | {model_name} | OOF epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | valid_auc_pr={valid_metrics['auc_pr']:.4f}"
        )

        if valid_metrics["auc_pr"] > best_valid_ap:
            best_valid_ap = valid_metrics["auc_pr"]
            best_epoch = epoch
            stale_epochs = 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            stale_epochs += 1

        if stale_epochs >= GNN_PATIENCE:
            print(f"Early stopping {model_name} OOF on {variant_name} at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    valid_metrics = evaluate_model(model, valid_loader)
    history_df = pd.DataFrame(history_rows)
    summary = {
        "variant_name": variant_name,
        "model_name": model_name,
        "best_epoch": int(best_epoch),
        "valid_auc_pr": float(valid_metrics["auc_pr"]),
        "valid_auc_roc": float(valid_metrics["auc_roc"]),
        "valid_recall_at_1pct_fpr": float(valid_metrics["recall_at_1pct_fpr"]),
    }
    del train_loader, valid_loader, model, optimizer, best_state
    release_memory()
    return history_df, valid_metrics["y_score"], summary

Raw pos_weight: 27.459
Clipped pos_weight for GNN training: 10.0


## Cell 14: Train Graph Models On Full Graph

In [16]:
rq1_jobs = [
    ("GCN", "full", "base"),
    ("GraphSAGE", "full", "base"),
    ("TemporalGraphSAGE", "full", "temporal"),
]

rq1_summary_rows = []
rq1_history_map = {}
rq1_predictions_df = phase1_predictions.copy()

for model_name, variant_name, feature_mode in rq1_jobs:
    summary_path = PHASE2_ROOT / f"{variant_name}_{model_name}_summary.json"
    history_path = PHASE2_ROOT / f"{variant_name}_{model_name}_history.csv"
    preds_path = PHASE2_ROOT / f"{variant_name}_{model_name}_predictions.parquet"

    if summary_path.exists() and history_path.exists() and preds_path.exists() and not REFRESH_MODELS:
        summary = load_json(summary_path)
        history_df = pd.read_csv(history_path)
        preds_df = pd.read_parquet(preds_path)
    else:
        history_df, preds_df, summary = fit_model(
            model_name=model_name,
            variant_name=variant_name,
            data_obj=data_objects[variant_name][feature_mode],
        )
        save_json(summary, summary_path)
        history_df.to_csv(history_path, index=False)
        preds_df.to_parquet(preds_path, index=False)

    rq1_summary_rows.append(summary)
    rq1_history_map[f"{variant_name}_{model_name}"] = history_df
    score_col = f"{variant_name}_{model_name}_score"
    rq1_predictions_df = rq1_predictions_df.merge(
        preds_df[[ID_COL, TIME_COL, score_col]],
        on=[ID_COL, TIME_COL],
        how="left",
    )

rq1_summary_df = pd.DataFrame(rq1_summary_rows).sort_values("test_auc_pr", ascending=False)
rq1_summary_df.to_csv(PHASE2_ROOT / "rq1_graph_model_summary.csv", index=False)
rq1_summary_df

full | GCN | epoch 01 | train_loss=0.4892 | valid_auc_pr=0.3435
full | GCN | epoch 02 | train_loss=0.4063 | valid_auc_pr=0.3757
full | GCN | epoch 03 | train_loss=0.3814 | valid_auc_pr=0.3743
full | GCN | epoch 04 | train_loss=0.3674 | valid_auc_pr=0.3897
full | GCN | epoch 05 | train_loss=0.3546 | valid_auc_pr=0.3916
full | GCN | epoch 06 | train_loss=0.3491 | valid_auc_pr=0.3981
full | GCN | epoch 07 | train_loss=0.3407 | valid_auc_pr=0.3979
full | GCN | epoch 08 | train_loss=0.3335 | valid_auc_pr=0.4095
full | GCN | epoch 09 | train_loss=0.3275 | valid_auc_pr=0.4096
full | GCN | epoch 10 | train_loss=0.3216 | valid_auc_pr=0.4030
full | GCN | epoch 11 | train_loss=0.3185 | valid_auc_pr=0.4153
full | GCN | epoch 12 | train_loss=0.3144 | valid_auc_pr=0.4169
full | GCN | epoch 13 | train_loss=0.3104 | valid_auc_pr=0.4237
full | GCN | epoch 14 | train_loss=0.3069 | valid_auc_pr=0.4199
full | GCN | epoch 15 | train_loss=0.3022 | valid_auc_pr=0.4203
full | GCN | epoch 16 | train_loss=0.299

,variant_name,model_name,best_epoch,valid_auc_pr,valid_auc_roc,valid_recall_at_1pct_fpr,test_auc_pr,test_auc_roc,test_recall_at_1pct_fpr,test_latency_ms,checkpoint_path
1,full,GraphSAGE,19,0.473944,0.866157,0.403570,0.436629,0.844807,0.369770,0.006379,/kaggle/working/thesis_outputs/artifacts/phase...
2,full,TemporalGraphSAGE,17,0.461059,0.866620,0.396133,0.418584,0.846526,0.351606,0.006532,/kaggle/working/thesis_outputs/artifacts/phase...
0,full,GCN,13,0.424887,0.847103,0.350025,0.396145,0.828461,0.328576,0.007542,/kaggle/working/thesis_outputs/artifacts/phase...


## Cell 14B: Temporal OOF Graph Predictions On Full Graph

In [17]:
GRAPH_OOF_PREDICTIONS_PATH = PHASE2_ROOT / "graph_oof_predictions.parquet"
GRAPH_OOF_SUMMARY_PATH = PHASE2_ROOT / "graph_oof_summary.json"

train_graph_df = graph_df[graph_df["split"] == "train"].copy().reset_index(drop=True)
train_processed_df = train_df.copy().reset_index(drop=True)

if GRAPH_OOF_PREDICTIONS_PATH.exists() and GRAPH_OOF_SUMMARY_PATH.exists() and not REFRESH_MODELS:
    graph_oof_predictions = pd.read_parquet(GRAPH_OOF_PREDICTIONS_PATH)
    graph_oof_summary = load_json(GRAPH_OOF_SUMMARY_PATH)
else:
    oof_rows = []
    oof_summary_rows = []
    oof_jobs = [
        ("GraphSAGE", "base"),
        ("TemporalGraphSAGE", "temporal"),
    ]

    for fold in oof_folds:
        train_end = int(fold["train_end"])
        eval_start = int(fold["eval_start"])
        eval_end = int(fold["eval_end"])
        fold_graph_df = train_graph_df.iloc[:eval_end].copy().reset_index(drop=True)
        fold_processed_df = train_processed_df.iloc[:eval_end].copy().reset_index(drop=True)
        fold_graph_df["split"] = "train"
        fold_graph_df.loc[train_end:, "split"] = "valid"

        fold_row = {
            "fold_id": int(fold["fold_id"]),
            "train_end": train_end,
            "eval_start": eval_start,
            "eval_end": eval_end,
        }
        fold_pred_df = pd.DataFrame(
            {
                "fold_id": int(fold["fold_id"]),
                "split": "train_oof",
                ID_COL: fold_graph_df.iloc[train_end:eval_end][ID_COL].to_numpy(),
                TIME_COL: fold_graph_df.iloc[train_end:eval_end][TIME_COL].to_numpy(),
                "y_true": fold_processed_df.iloc[train_end:eval_end][TARGET_COL].astype(int).to_numpy(),
            }
        )

        for model_name, feature_mode in oof_jobs:
            print(
                f"Phase 2 OOF fold {fold['fold_id']} | {model_name}: "
                f"train [0:{train_end}) -> predict [{eval_start}:{eval_end})"
            )
            fold_data_obj, _, _ = build_data_object_from_subset(
                subset_graph_df=fold_graph_df,
                subset_processed_df=fold_processed_df,
                variant_name="full",
                feature_mode=feature_mode,
                train_rows_count=train_end,
            )
            history_df, fold_scores, summary = fit_oof_graph_model(
                model_name=model_name,
                variant_name=f"full_oof_fold_{fold['fold_id']}",
                data_obj=fold_data_obj,
            )
            score_col = f"full_{model_name}_score"
            fold_pred_df[score_col] = fold_scores
            fold_row[f"{model_name}_auc_pr"] = summary["valid_auc_pr"]
            fold_row[f"{model_name}_auc_roc"] = summary["valid_auc_roc"]
            history_df.to_csv(
                PHASE2_ROOT / f"full_oof_fold_{fold['fold_id']}_{model_name}_history.csv",
                index=False,
            )
            del fold_data_obj, history_df, fold_scores
            release_memory()

        oof_rows.append(fold_pred_df)
        oof_summary_rows.append(fold_row)
        del fold_graph_df, fold_processed_df, fold_pred_df
        release_memory()

    graph_oof_predictions = pd.concat(oof_rows, axis=0, ignore_index=True).sort_values(TIME_COL).reset_index(drop=True)
    graph_oof_summary = {
        "variant_name": "full",
        "rows_covered": int(len(graph_oof_predictions)),
        "coverage_fraction_of_train": float(len(graph_oof_predictions) / max(len(train_df), 1)),
        "folds": oof_summary_rows,
        "overall_graphsage_auc_pr": float(
            average_precision_score(graph_oof_predictions["y_true"], graph_oof_predictions["full_GraphSAGE_score"])
        ),
        "overall_temporal_graphsage_auc_pr": float(
            average_precision_score(
                graph_oof_predictions["y_true"],
                graph_oof_predictions["full_TemporalGraphSAGE_score"],
            )
        ),
    }
    graph_oof_predictions.to_parquet(GRAPH_OOF_PREDICTIONS_PATH, index=False)
    save_json(graph_oof_summary, GRAPH_OOF_SUMMARY_PATH)

graph_oof_summary

Phase 2 OOF fold 1 | GraphSAGE: train [0:177162) -> predict [177162:265743)
full_oof_fold_1 | GraphSAGE | OOF epoch 01 | train_loss=0.3990 | valid_auc_pr=0.4212
full_oof_fold_1 | GraphSAGE | OOF epoch 02 | train_loss=0.3096 | valid_auc_pr=0.4249
full_oof_fold_1 | GraphSAGE | OOF epoch 03 | train_loss=0.2921 | valid_auc_pr=0.4434
full_oof_fold_1 | GraphSAGE | OOF epoch 04 | train_loss=0.2774 | valid_auc_pr=0.4489
full_oof_fold_1 | GraphSAGE | OOF epoch 05 | train_loss=0.2678 | valid_auc_pr=0.4493
full_oof_fold_1 | GraphSAGE | OOF epoch 06 | train_loss=0.2552 | valid_auc_pr=0.4525
full_oof_fold_1 | GraphSAGE | OOF epoch 07 | train_loss=0.2453 | valid_auc_pr=0.4403
full_oof_fold_1 | GraphSAGE | OOF epoch 08 | train_loss=0.2382 | valid_auc_pr=0.4405
full_oof_fold_1 | GraphSAGE | OOF epoch 09 | train_loss=0.2327 | valid_auc_pr=0.4389
full_oof_fold_1 | GraphSAGE | OOF epoch 10 | train_loss=0.2277 | valid_auc_pr=0.4399
full_oof_fold_1 | GraphSAGE | OOF epoch 11 | train_loss=0.2203 | valid_auc

{'variant_name': 'full',
 'rows_covered': 265743,
 'coverage_fraction_of_train': 0.6,
 'folds': [{'fold_id': 1,
   'train_end': 177162,
   'eval_start': 177162,
   'eval_end': 265743,
   'GraphSAGE_auc_pr': 0.4525704534118978,
   'GraphSAGE_auc_roc': 0.8030263856257669,
   'TemporalGraphSAGE_auc_pr': 0.45672586489335076,
   'TemporalGraphSAGE_auc_roc': 0.791947537840776},
  {'fold_id': 2,
   'train_end': 265743,
   'eval_start': 265743,
   'eval_end': 354324,
   'GraphSAGE_auc_pr': 0.40042921936074755,
   'GraphSAGE_auc_roc': 0.8239288847906775,
   'TemporalGraphSAGE_auc_pr': 0.41741385152530597,
   'TemporalGraphSAGE_auc_roc': 0.8231157524461312},
  {'fold_id': 3,
   'train_end': 354324,
   'eval_start': 354324,
   'eval_end': 442905,
   'GraphSAGE_auc_pr': 0.4939973569739307,
   'GraphSAGE_auc_roc': 0.8663692451820348,
   'TemporalGraphSAGE_auc_pr': 0.49786231996715236,
   'TemporalGraphSAGE_auc_roc': 0.8648405320299068}],
 'overall_graphsage_auc_pr': 0.44799156299641946,
 'overall_t

## Cell 15: Graph Variant Ablation With TemporalGraphSAGE

In [18]:
rq5_rows = []

if RUN_GRAPH_VARIANT_ABLATION:
    for variant_name in graph_variant_specs:
        summary_path = PHASE2_ROOT / f"{variant_name}_TemporalGraphSAGE_rq5_summary.json"
        preds_path = PHASE2_ROOT / f"{variant_name}_TemporalGraphSAGE_rq5_predictions.parquet"

        if variant_name == "full" and (PHASE2_ROOT / "full_TemporalGraphSAGE_summary.json").exists() and not REFRESH_MODELS:
            summary = load_json(PHASE2_ROOT / "full_TemporalGraphSAGE_summary.json")
        elif summary_path.exists() and preds_path.exists() and not REFRESH_MODELS:
            summary = load_json(summary_path)
        else:
            data_obj = build_data_object_for_variant(variant_name, "temporal")
            _, preds_df, summary = fit_model(
                model_name="TemporalGraphSAGE",
                variant_name=variant_name,
                data_obj=data_obj,
            )
            preds_df.to_parquet(preds_path, index=False)
            save_json(summary, summary_path)
            del data_obj, preds_df
            release_memory()

        meta = load_json(PHASE2_ROOT / f"{variant_name}_metadata.json")
        rq5_rows.append(
            {
                "GraphType": GRAPH_DISPLAY_NAMES[variant_name],
                "VariantName": variant_name,
                "Model": "TemporalGraphSAGE",
                "AUC-PR": summary["test_auc_pr"],
                "AUC-ROC": summary["test_auc_roc"],
                "Recall@1%FPR": summary["test_recall_at_1pct_fpr"],
                "Nodes": meta["num_nodes_total"],
                "Edges": meta["num_edges"],
                "AvgDegree": meta["avg_degree"],
            }
        )

if rq5_rows:
    rq5_df = pd.DataFrame(rq5_rows).sort_values("AUC-PR", ascending=False)
else:
    rq5_df = pd.DataFrame(
        columns=[
            "GraphType",
            "VariantName",
            "Model",
            "AUC-PR",
            "AUC-ROC",
            "Recall@1%FPR",
            "Nodes",
            "Edges",
            "AvgDegree",
        ]
    )
rq5_df

cards | TemporalGraphSAGE | epoch 01 | train_loss=0.4357 | valid_auc_pr=0.4167
cards | TemporalGraphSAGE | epoch 02 | train_loss=0.3737 | valid_auc_pr=0.4395
cards | TemporalGraphSAGE | epoch 03 | train_loss=0.3548 | valid_auc_pr=0.4490
cards | TemporalGraphSAGE | epoch 04 | train_loss=0.3390 | valid_auc_pr=0.4555
cards | TemporalGraphSAGE | epoch 05 | train_loss=0.3296 | valid_auc_pr=0.4592
cards | TemporalGraphSAGE | epoch 06 | train_loss=0.3196 | valid_auc_pr=0.4790
cards | TemporalGraphSAGE | epoch 07 | train_loss=0.3096 | valid_auc_pr=0.4831
cards | TemporalGraphSAGE | epoch 08 | train_loss=0.3024 | valid_auc_pr=0.4705
cards | TemporalGraphSAGE | epoch 09 | train_loss=0.2950 | valid_auc_pr=0.4798
cards | TemporalGraphSAGE | epoch 10 | train_loss=0.2883 | valid_auc_pr=0.4898
cards | TemporalGraphSAGE | epoch 11 | train_loss=0.2832 | valid_auc_pr=0.5009
cards | TemporalGraphSAGE | epoch 12 | train_loss=0.2788 | valid_auc_pr=0.4905
cards | TemporalGraphSAGE | epoch 13 | train_loss=0.

,GraphType,VariantName,Model,AUC-PR,AUC-ROC,Recall@1%FPR,Nodes,Edges,AvgDegree
0,Cards,cards,TemporalGraphSAGE,0.460216,0.865918,0.383068,601305,1172920,1.963950
1,Cards+Merchants,cards_plus_merchants,TemporalGraphSAGE,0.446542,0.864159,0.367175,601738,1892198,3.151457
2,Full,full,TemporalGraphSAGE,0.418584,0.846526,0.351606,603285,2254186,3.740847


## Cell 16: Temporal Decay Ablation On Full Graph

In [19]:
decay_rows = []

if RUN_TEMPORAL_DECAY_ABLATION:
    for decay_lambda in [0.0, 0.01, 0.02, 0.05]:
        variant_name = f"full_decay_{str(decay_lambda).replace('.', '_')}"
        edge_file = PHASE2_ROOT / f"{variant_name}_edges.parquet"
        entity_file = PHASE2_ROOT / f"{variant_name}_entity_nodes.parquet"
        meta_file = PHASE2_ROOT / f"{variant_name}_metadata.json"
        summary_path = PHASE2_ROOT / f"{variant_name}_TemporalGraphSAGE_summary.json"

        if edge_file.exists() and entity_file.exists() and meta_file.exists() and not REFRESH_GRAPHS:
            edge_df = pd.read_parquet(edge_file)
            entity_df = pd.read_parquet(entity_file)
        else:
            edge_df, entity_df = build_heterogeneous_graph(
                graph_source_df=graph_df,
                variant_name=variant_name,
                relation_specs=graph_variant_specs["full"],
                decay_lambda=decay_lambda,
            )
            edge_df.to_parquet(edge_file, index=False)
            entity_df.to_parquet(entity_file, index=False)
            degree_series = edge_df.groupby("src").size()
            save_json(
                {
                    "variant_name": variant_name,
                    "graph_type": "Full",
                    "num_nodes_total": int(len(graph_df) + len(entity_df)),
                    "num_edges": int(len(edge_df)),
                    "avg_degree": float(degree_series.mean()),
                    "decay_lambda": decay_lambda,
                },
                meta_file,
            )

        variant_graphs[variant_name] = {
            "edge_path": str(edge_file),
            "entity_path": str(entity_file),
            "metadata_path": str(meta_file),
        }
        data_obj = build_data_object_for_variant(variant_name, "temporal")

        if summary_path.exists() and not REFRESH_MODELS:
            summary = load_json(summary_path)
        else:
            _, _, summary = fit_model(
                model_name="TemporalGraphSAGE",
                variant_name=variant_name,
                data_obj=data_obj,
            )
            save_json(summary, summary_path)
        del data_obj, edge_df, entity_df
        release_memory()

        decay_rows.append(
            {
                "Lambda": decay_lambda,
                "AUC-PR": summary["test_auc_pr"],
                "AUC-ROC": summary["test_auc_roc"],
                "Recall@1%FPR": summary["test_recall_at_1pct_fpr"],
            }
        )

if decay_rows:
    decay_df = pd.DataFrame(decay_rows).sort_values("Lambda")
else:
    decay_df = pd.DataFrame(
        columns=[
            "Lambda",
            "AUC-PR",
            "AUC-ROC",
            "Recall@1%FPR",
        ]
    )
decay_df

full_decay_0_0 | TemporalGraphSAGE | epoch 01 | train_loss=0.4277 | valid_auc_pr=0.3969
full_decay_0_0 | TemporalGraphSAGE | epoch 02 | train_loss=0.3676 | valid_auc_pr=0.4150
full_decay_0_0 | TemporalGraphSAGE | epoch 03 | train_loss=0.3483 | valid_auc_pr=0.4263
full_decay_0_0 | TemporalGraphSAGE | epoch 04 | train_loss=0.3347 | valid_auc_pr=0.4361
full_decay_0_0 | TemporalGraphSAGE | epoch 05 | train_loss=0.3229 | valid_auc_pr=0.4469
full_decay_0_0 | TemporalGraphSAGE | epoch 06 | train_loss=0.3139 | valid_auc_pr=0.4505
full_decay_0_0 | TemporalGraphSAGE | epoch 07 | train_loss=0.3048 | valid_auc_pr=0.4512
full_decay_0_0 | TemporalGraphSAGE | epoch 08 | train_loss=0.2970 | valid_auc_pr=0.4522
full_decay_0_0 | TemporalGraphSAGE | epoch 09 | train_loss=0.2923 | valid_auc_pr=0.4664
full_decay_0_0 | TemporalGraphSAGE | epoch 10 | train_loss=0.2853 | valid_auc_pr=0.4645
full_decay_0_0 | TemporalGraphSAGE | epoch 11 | train_loss=0.2793 | valid_auc_pr=0.4762
full_decay_0_0 | TemporalGraphSA

,Lambda,AUC-PR,AUC-ROC,Recall@1%FPR
0,0.00,0.450463,0.862738,0.374959
1,0.01,0.424628,0.840385,0.357120
2,0.02,0.443873,0.857957,0.366202
3,0.05,0.435398,0.858563,0.362958


## Cell 17: Save Raw RQ1 Support Exports
#
This phase now saves only raw graph outputs.
Proposal-facing RQ1 and RQ5 figures/tables must be generated separately.

In [20]:
table_1_1_rows = [
    {
        "Model": "LightGBM",
        "AUC-ROC": phase1_metrics["test_lgbm_auc_roc"],
        "AUC-PR": phase1_metrics["test_lgbm_auc_pr"],
        "Recall@1%FPR": phase1_metrics["test_lgbm_recall_at_1pct_fpr"],
        "Latency(ms)": phase1_metrics.get("test_lgbm_latency_ms", np.nan),
    },
    {
        "Model": "XGBoost",
        "AUC-ROC": phase1_metrics["test_xgb_auc_roc"],
        "AUC-PR": phase1_metrics["test_xgb_auc_pr"],
        "Recall@1%FPR": phase1_metrics["test_xgb_recall_at_1pct_fpr"],
        "Latency(ms)": phase1_metrics.get("test_xgb_latency_ms", np.nan),
    },
]

for _, row in rq1_summary_df.iterrows():
    if row["variant_name"] != "full":
        continue
    display_name = row["model_name"]
    table_1_1_rows.append(
        {
            "Model": display_name,
            "AUC-ROC": row["test_auc_roc"],
            "AUC-PR": row["test_auc_pr"],
            "Recall@1%FPR": row["test_recall_at_1pct_fpr"],
            "Latency(ms)": row.get("test_latency_ms", np.nan),
        }
    )

rq1_comparison_raw = pd.DataFrame(table_1_1_rows)
rq1_comparison_raw.to_csv(PHASE2_ROOT / "rq1_performance_comparison_raw.csv", index=False)

test_prediction_df = rq1_predictions_df[rq1_predictions_df["split"] == "test"].copy().reset_index(drop=True)
y_test = test_prediction_df["y_true"].to_numpy()

pr_curve_rows = []
curve_specs = [
    ("LightGBM", "lgbm_score"),
    ("XGBoost", "xgb_score"),
]
for _, row in rq1_summary_df.iterrows():
    if row["variant_name"] != "full":
        continue
    curve_specs.append((row["model_name"], f"{row['variant_name']}_{row['model_name']}_score"))

for model_name, score_col in curve_specs:
    precision, recall, _ = precision_recall_curve(y_test, test_prediction_df[score_col].to_numpy())
    pr_curve_rows.append(pd.DataFrame({"Model": model_name, "Recall": recall, "Precision": precision}))

pd.concat(pr_curve_rows, ignore_index=True).to_csv(PHASE2_ROOT / "rq1_precision_recall_curve_points.csv", index=False)

history_index_rows = []
for label in sorted(rq1_history_map.keys()):
    history_index_rows.append(
        {
            "label": label,
            "history_path": str(PHASE2_ROOT / f"{label}_history.csv"),
        }
    )
pd.DataFrame(history_index_rows).to_csv(PHASE2_ROOT / "rq1_history_index.csv", index=False)

budget_rows = []
for model_name, score_col in [
    ("LightGBM", "lgbm_score"),
    ("XGBoost", "xgb_score"),
    ("GCN", "full_GCN_score"),
    ("GraphSAGE", "full_GraphSAGE_score"),
    ("TemporalGraphSAGE", "full_TemporalGraphSAGE_score"),
]:
    for budget in ANALYST_BUDGETS:
        budget_rows.append(
            {
                "Budget": budget,
                "Model": model_name,
                "Recall": recall_at_k(y_test, test_prediction_df[score_col].to_numpy(), budget),
            }
        )

budget_df = pd.DataFrame(budget_rows)
budget_df.to_csv(PHASE2_ROOT / "rq1_recall_vs_budget_raw.csv", index=False)

## Cell 18: Save Raw RQ5 Support Exports

In [21]:
rq5_df.to_csv(PHASE2_ROOT / "rq5_graph_variant_summary.csv", index=False)
decay_df.to_csv(PHASE2_ROOT / "rq5_temporal_decay_summary.csv", index=False)

full_edges_df = pd.read_parquet(variant_graphs["full"]["edge_path"])
degree_df = full_edges_df.groupby("src").size().reset_index(name="degree")
degree_df = degree_df[degree_df["degree"] > 0].copy()
degree_count_df = (
    degree_df["degree"]
    .value_counts()
    .sort_index()
    .reset_index()
)
degree_count_df.columns = ["degree", "frequency"]
degree_count_df.to_csv(PHASE2_ROOT / "full_degree_distribution_raw.csv", index=False)
del full_edges_df, degree_df, degree_count_df
release_memory()

## Cell 19: Save Master Outputs

In [22]:
rq1_predictions_df.to_parquet(PHASE2_ROOT / "rq1_predictions.parquet", index=False)

phase2_manifest = {
    "phase": "phase02_graph_models",
    "device": DEVICE,
    "files": {
        "rq1_predictions": str(PHASE2_ROOT / "rq1_predictions.parquet"),
        "rq1_graph_summary": str(PHASE2_ROOT / "rq1_graph_model_summary.csv"),
        "graph_oof_predictions": str(GRAPH_OOF_PREDICTIONS_PATH),
        "graph_oof_summary": str(GRAPH_OOF_SUMMARY_PATH),
        "rq1_performance_comparison_raw": str(PHASE2_ROOT / "rq1_performance_comparison_raw.csv"),
        "rq1_precision_recall_curve_points": str(PHASE2_ROOT / "rq1_precision_recall_curve_points.csv"),
        "rq1_history_index": str(PHASE2_ROOT / "rq1_history_index.csv"),
        "rq1_recall_vs_budget_raw": str(PHASE2_ROOT / "rq1_recall_vs_budget_raw.csv"),
        "rq5_graph_variant_summary": str(PHASE2_ROOT / "rq5_graph_variant_summary.csv"),
        "rq5_temporal_decay_summary": str(PHASE2_ROOT / "rq5_temporal_decay_summary.csv"),
        "graph_variant_metadata": str(PHASE2_ROOT / "graph_variant_metadata.csv"),
        "full_degree_distribution_raw": str(PHASE2_ROOT / "full_degree_distribution_raw.csv"),
    },
}
save_json(phase2_manifest, OUTPUT_ROOT / "manifests" / "phase02_artifact_manifest.json")
phase2_manifest

{'phase': 'phase02_graph_models',
 'device': 'cuda',
 'files': {'rq1_predictions': '/kaggle/working/thesis_outputs/artifacts/phase02_graph_models/rq1_predictions.parquet',
  'rq1_graph_summary': '/kaggle/working/thesis_outputs/artifacts/phase02_graph_models/rq1_graph_model_summary.csv',
  'graph_oof_predictions': '/kaggle/working/thesis_outputs/artifacts/phase02_graph_models/graph_oof_predictions.parquet',
  'graph_oof_summary': '/kaggle/working/thesis_outputs/artifacts/phase02_graph_models/graph_oof_summary.json',
  'rq1_performance_comparison_raw': '/kaggle/working/thesis_outputs/artifacts/phase02_graph_models/rq1_performance_comparison_raw.csv',
  'rq1_precision_recall_curve_points': '/kaggle/working/thesis_outputs/artifacts/phase02_graph_models/rq1_precision_recall_curve_points.csv',
  'rq1_history_index': '/kaggle/working/thesis_outputs/artifacts/phase02_graph_models/rq1_history_index.csv',
  'rq1_recall_vs_budget_raw': '/kaggle/working/thesis_outputs/artifacts/phase02_graph_model

## Cell 20: Hand-Off To Notebook 03
#
Notebook 03 should read:
- `rq1_predictions.parquet`
- graph metadata and edge files
- best graph model checkpoints
- graph summary tables

In [23]:
gc.collect()
print("Phase 02 finished.")
print("Phase 02 now saves raw graph outputs only.")
print("Generate proposal-facing RQ1 and RQ5 artifacts separately after Phase 03 is stabilized.")

Phase 02 finished.
Phase 02 now saves raw graph outputs only.
Generate proposal-facing RQ1 and RQ5 artifacts separately after Phase 03 is stabilized.
